# Singular Value Decomposition and Covariance Analysis

This notebook contains all code examples from Lesson 12.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler # sklearn is scikit-learn
from sklearn.decomposition import TruncatedSVD

## 0. Diagonalization Example

In [2]:
# Simple 2x2 example
A = np.array([[4, -1], 
              [-1, 4]])

# Find eigenvalues and eigenvectors
eigvals, eigvecs = np.linalg.eig(A)

# Form P and D
P = eigvecs
D = np.diag(eigvals)

# Verify decomposition
print("A = PDP^(-1):", np.allclose(A, P @ D @ np.linalg.inv(P)))
print("Av = λv:", np.allclose(A @ eigvecs[:, 0], eigvals[0] * eigvecs[:, 0]))

A = PDP^(-1): True
Av = λv: True


## 1. SVD Implementation Examples

In [ ]:
# Original data
X = np.array([[1000, 2], [2000, 4], [3000, 6]])

# No scaling
U1, s1, Vh1 = np.linalg.svd(X)
print("Original singular values:", s1)

# With standardization
X_std = (X - X.mean(axis=0)) / X.std(axis=0)
U2, s2, Vh2 = np.linalg.svd(X_std)
print("\nStandardized singular values:", s2)

# With robust scaling
X_rob = RobustScaler().fit_transform(X)
U3, s3, Vh3 = np.linalg.svd(X_rob)
print("\nRobust scaled singular values:", s3)

In [ ]:
# Verify properties
for i, (U, s, Vh) in enumerate([(U1, s1, Vh1), (U2, s2, Vh2), (U3, s3, Vh3)]):
    print(f"\nScaling method {i+1}:")
    # Check orthogonality
    print("U orthogonal:", np.allclose(U.T @ U, np.eye(U.shape[1])))
    print("V orthogonal:", np.allclose(Vh @ Vh.T, np.eye(Vh.shape[0])))
    
    # Check reconstruction
    X_check = X if i == 0 else (X_std if i == 1 else X_rob)
    print("Reconstruction:", np.allclose(U @ np.diag(s) @ Vh, X_check))
    
    # Additional validations
    print("Singular values sorted:", np.all(s[:-1] >= s[1:]))
    print("Singular values non-negative:", np.all(s >= 0))

## 2. Random Data Example

In [ ]:
# Generate random data with known covariance structure
rng = np.random.default_rng(42)
n_samples = 1000
X = rng.multivariate_normal(
    mean=[0, 0],
    cov=[[4, 1], [1, 1]],
    size=n_samples
)

# Compute SVD
U, s, Vh = np.linalg.svd(X - X.mean(axis=0))

# Compare singular values with theoretical values
print("Singular values:", s)
print("Expected stdevs:", np.sqrt([4, 1]))  # sqrt of eigenvalues

## 3. Iris Dataset Example

In [3]:
# Load iris dataset
from sklearn.datasets import load_iris
X = load_iris().data
feature_names = load_iris().feature_names

def no_scaling(x):
    return x

# Compare SVD with different scaling methods
methods = {
    'none': no_scaling,
    'standard': StandardScaler().fit_transform,
    'robust': RobustScaler().fit_transform
}

results = {}
for name, scaler in methods.items():
    X_scaled = scaler(X)
    U, s, Vh = np.linalg.svd(X_scaled)
    results[name] = {
        'singular_values': s,
        'explained_variance_ratio': s**2/sum(s**2),
        'feature_importance': np.abs(Vh[0])  # First component loadings
    }

# Analyze how scaling affects interpretation
for name, res in results.items():
    print(f"\n{name.title()} Scaling:")
    print("Singular values:", res['singular_values'])
    print("Variance explained:", res['explained_variance_ratio'])
    print("Feature importance:")
    for feat, imp in zip(feature_names, res['feature_importance']):
        print(f"  {feat}: {imp:.3f}")


None Scaling:
Singular values: [95.95991387 17.76103366  3.46093093  1.88482631]
Variance explained: [9.65302981e-01 3.30689513e-02 1.25565350e-03 3.72414530e-04]
Feature importance:
  sepal length (cm): 0.751
  sepal width (cm): 0.380
  petal length (cm): 0.513
  petal width (cm): 0.168

Standard Scaling:
Singular values: [20.92306556 11.7091661   4.69185798  1.76273239]
Variance explained: [0.72962445 0.22850762 0.03668922 0.00517871]
Feature importance:
  sepal length (cm): 0.521
  sepal width (cm): 0.269
  petal length (cm): 0.580
  petal width (cm): 0.565

Robust Scaling:
Singular values: [12.61334184  9.33576419  2.985075    1.41575593]
Variance explained: [0.61864789 0.33890888 0.03464924 0.00779399]
Feature importance:
  sepal length (cm): 0.421
  sepal width (cm): 0.657
  petal length (cm): 0.461
  petal width (cm): 0.424


In [4]:
load_iris().target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [ ]:
"setosaness"
"versicolorness"